# 03 — Sampling: temperature, top_p, top_k

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase02/notebooks/03_sampling_params.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objetivo.** Tocar los parámetros de sampling y ver cómo cambia el output. La idea es que después puedas elegirlos a conciencia para tu caso de uso.

**Requisitos.** API key de Groq en `GROQ_API_KEY`.


In [1]:
%pip install  groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.2 MB/s eta 0:00:00


In [2]:
import os
from groq import Groq

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    assert os.environ.get("GROQ_API_KEY"), "Exportá GROQ_API_KEY."

client = Groq()
MODELS = {
    "llama_fast":   "llama-3.1-8b-instant",
    "llama_strong": "llama-3.3-70b-versatile",
    "qwen_reason":  "qwen/qwen3-32b",
    "deepseek":     "deepseek-r1-distill-llama-70b",
    "gemma":        "gemma2-9b-it",
}
MODEL = MODELS["qwen_reason"]  # cambiá la clave para probar otros modelos

def generar(prompt, **kwargs):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        **kwargs,
    )
    return resp.choices[0].message.content


## 1. `temperature` — el termostato de la creatividad

Mismo prompt, distintas temperaturas. Observá la diferencia de tono y vocabulario.


In [3]:
PROMPT = "Escribime un poema corto (4 versos) sobre el otoño en Santa Fe de la vera cruz, de la provincia invencible de santa fe, mejor que entre rios."

for temp in [0.0, 0.3, 0.7, 1.2]:
    print(f"--- temperature = {temp} ---")
    print(generar(PROMPT, temperature=temp))
    print()


--- temperature = 0.0 ---
En Santa Fe de la Vera Cruz, donde el río Paraná fluye,
El otoño llega con suaves brisas y colores que seducen.
La ciudad se viste de tonos dorados y rojizos,
Y el clima agradable invita a disfrutar de su encanto.

--- temperature = 0.3 ---
En Santa Fe de la Vera Cruz, donde el río Paraná se desliza,
El otoño llega con suaves brisas y colores que se desvanecen.
La ciudad se viste de dorado y rojo, en un espectáculo sin igual,
En la provincia invencible de Santa Fe, el otoño es un regalo celestial.

--- temperature = 0.7 ---
En Santa Fe de la Vera Cruz, ciudad de encanto,
El otoño llega con suaves vientos de cambio,
Las hojas doradas caen en el río,
Y el paisaje se vuelve un cuadro de otoño tranquilo y encantado.

--- temperature = 1.2 ---
En Santa Fe de la Vera Cruz, el otoño llega
Con colores cálidos, suaves y serenos que bañan
La ciudad invencible, con historias que cuentan
Y el río Paraná, que la acompañan.



- `temperature=0` → el modelo elige siempre el token más probable. Determinista.
- Valores altos → distribución más plana, salidas diversas (y a veces incoherentes).


## 2. Reproducibilidad: el bug del "siempre lo mismo"

Con temperatura baja, dos llamadas seguidas dan respuestas casi idénticas. Útil cuando necesitás determinismo (testing, código).


In [ ]:
PROMPT = "Listame 3 razones por las que se prefiere PostgreSQL sobre MySQL."

for i in range(2):
    print(f"--- Llamada {i+1} (temperature=0) ---")
    print(generar(PROMPT, temperature=0))
    print()


## 3. `top_p` — nucleus sampling

Muestrea solo del conjunto de tokens cuya probabilidad acumulada llega a P. Recorta la "cola larga".


In [ ]:
PROMPT = "Inventame el nombre de una banda de rock progresivo argentina."

for p in [0.1, 0.5, 1.0]:
    print(f"--- top_p = {p} ---")
    for _ in range(3):
        print("  ·", generar(PROMPT, temperature=0.9, top_p=p))
    print()


## Cuándo usar qué

| Caso | temperature | top_p |
|---|---|---|
| Código, factual, extracción | 0.0 – 0.3 | 1.0 |
| Conversación natural | 0.6 – 0.8 | 0.9 |
| Creatividad, brainstorming | 0.9 – 1.2 | 0.95 |
| Determinismo (tests) | 0.0 | 1.0 |
